<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_6_Implementing_a_Toy_Transformer_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 6: Implementing a Toy Transformer from Scratch


Instructor: John Sipple

### Overview

This tutorial marks the transition from standard neural networks to the architecture powering modern Large Language Models (LLMs). Rather than using pre-built library layers, this notebook guides you through constructing a generative, decoder-only ["Nano-GPT"](https://bbycroft.net/llm?utm_source=theprompt.io&utm_medium=referral&utm_campaign=3d-llm-visualization) model entirely from scratch using JAX and Equinox. By building a character-level tokenizer and training it on a small dataset of Aesop's Fables, the tutorial demystifies how models learn to autoregressively predict the probability of the next character in a sequence.

### Key Concepts Explored:

*
**Activation Functions (ReLU vs. GELU):** The notebook mathematically compares the traditional Rectified Linear Unit (ReLU) with the Gaussian Error Linear Unit (GELU). It visualizes how GELU's smooth, differentiable curve and "negative dip" help prevent the "dying neuron" problem and allow better gradient flow in deep architectures.


*
**Causal Self-Attention:** You will implement the core attention mechanism by calculating Queries, Keys, and Values. The tutorial demonstrates how to apply a lower-triangular mask of negative infinity to the attention scores, strictly preventing a token from "looking into the future" during text generation.


*
**Parameter Mechanics & Dimensionality:** The tutorial breaks down exactly how learnable parameters are distributed across the network. It calculates the mathematical sizes for the Token and Position Embeddings, LayerNorms, Attention projections, and the expansion/contraction phases of the Multi-Layer Perceptron (MLP).


*
**Dynamic Visualizations:** The notebook includes utilities to introspect the JAX PyTree and dynamically render high-fidelity blueprints of the Transformer block, visually mapping how tensor shapes transform as they flow through the attention and MLP residual streams.


*
**Training and Autoregressive Generation:** It implements a custom training loop using Optax (AdamW) and batched cross-entropy loss, heavily relying on `jax.vmap` to efficiently process sequences. The trained model is then used to generate interactive text by predicting the next token, appending it to the context, and repeating.


*
**Mechanistic Interpretability & Hallucinations:** The final sections explore why AI models hallucinate, explaining them as probabilistic engines interpolating in latent space. You will extract and plot intermediate Attention Maps to see exactly what characters the model focuses on, and visualize the internal MLP activations to see which specific "memory" neurons fire for specific concepts.



### Real-World Application

This tutorial strips away the "black box" nature of modern LLMs. Understanding the exact mechanics of attention bottlenecks and MLP dimensionality is a necessary foundation for engineers who need to modify sequence models for non-text domains (like genomics or log-parsing), debug model hallucinations, or optimize large-scale networks for efficient inference.

In [ ]:
# --- Setup and Installation ---
!pip install -q equinox optax jax jaxlib matplotlib numpy

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set precision for better gradient stability
jax.config.update("jax_enable_x64", False)

print(f"JAX Devices: {jax.devices()}")
print(f"Equinox Version: {eqx.__version__}")

## Part 1a: A Trivial Language Dataset

To train our model quickly, we will use a small toy dataset: a short collection of *Aesop's Fables*. We will build a **character-level tokenizer**.

Our model will learn the probability distribution of the next character given the previous characters:

$$P(x_t | x_{t-1}, x_{t-2}, \dots, x_{t-n})$$

In [ ]:
# --- 1. Toy Dataset ---
text = """
The Hare and the Tortoise.
A Hare was making fun of the Tortoise one day for being so slow.
"Do you ever get anywhere?" he asked with a mocking laugh.
"Yes," replied the Tortoise, "and I get there sooner than you think. I'll run you a race and prove it."
The Hare was much amused at the idea of running a race with the Tortoise, but for the fun of the thing he agreed.
So the Fox, who had consented to act as judge, marked the distance and started the runners off.
The Hare was soon far out of sight, and to make the Tortoise feel very deeply how ridiculous it was for him to try a race with a Hare, he lay down beside the course to take a nap until the Tortoise should catch up.
The Tortoise meanwhile kept going slowly but steadily, and, after a time, passed the place where the Hare was sleeping. But the Hare slept on very peacefully; and when at last he did wake up, the Tortoise was near the goal.
The Hare now ran his swiftest, but he could not overtake the Tortoise in time.
"""

# --- 2. Character-level Tokenizer ---
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary Size: {vocab_size} unique characters")
print(f"Encoded 'Tortoise': {encode('Tortoise')}")

# --- 3. Create Sequences ---
data = jnp.array(encode(text))
SEQ_LEN = 32

def create_batches(data, seq_len):
    # X is the input sequence, Y is the target sequence (shifted by 1)
    X, Y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        Y.append(data[i+1:i+seq_len+1])
    return jnp.array(X), jnp.array(Y)

X_all, Y_all = create_batches(data, SEQ_LEN)

# Train/Test Split (90/10)
split = int(0.9 * len(X_all))
X_train, Y_train = X_all[:split], Y_all[:split]
X_val, Y_val = X_all[split:], Y_all[split:]

print(f"Train shapes: X={X_train.shape}, Y={Y_train.shape}")

In [ ]:
itos

##Part 1b: The Non-Linearity - Why GELU instead of ReLU?

Before we build the Transformer block, we need to discuss the activation function used inside the Multi-Layer Perceptron (MLP). For years, the **Rectified Linear Unit (ReLU)** was the undisputed king of deep learning. However, modern Transformers (like BERT and GPT) have almost universally adopted the **Gaussian Error Linear Unit (GELU)**.

###Mathematical Definitions

####**ReLU (Rectified Linear Unit)**:
The standard gating mechanism. It simply zeroes out negative values and passes positive values linearly.

$$\text{ReLU}(x) = \max(0, x)$$

*The problem*: ReLU has a sharp, non-differentiable corner at $x = 0$, and its derivative is exactly $0$ for all negative inputs. This can lead to "dying neurons" where gradients stop flowing completely.

####**GELU (Gaussian Error Linear Unit)**:

GELU weights the input $x$ by its percentile in a standard normal distribution. It mathematically models the idea of a probabilistic dropout.

$$\text{GELU}(x) = x \cdot \Phi(x)$$

(Where $\Phi(x)$ is the cumulative distribution function of the Gaussian distribution).

####Why use GELU in Transformers?

* **Smoothness:** Unlike ReLU, GELU is completely smooth and differentiable everywhere. In the massively deep and complex optimization landscape of a Transformer, this smoothness allows for better gradient flow.

* **Negative Values:** GELU allows a small amount of negative information to pass through before curving back up to zero. This "negative dip" provides a more nuanced gating mechanism than ReLU's hard cutoff, allowing the network to retain context about how negative a pre-activation was.


Let's visualize them side-by-side using JAX to see the difference, especially around the origin.

In [ ]:
import jax.numpy as jnp
import jax.nn
import matplotlib.pyplot as plt

# Define a range of inputs
x = jnp.linspace(-3, 3, 500)

# Calculate activations using JAX
y_relu = jax.nn.relu(x)
y_gelu = jax.nn.gelu(x)

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Macro View
axes[0].plot(x, y_relu, label="ReLU", color="blue", linestyle="--", linewidth=2)
axes[0].plot(x, y_gelu, label="GELU", color="orange", linewidth=2)
axes[0].set_title("Macro View: ReLU vs GELU", fontsize=14)
axes[0].set_xlabel("x")
axes[0].set_ylabel("Activation f(x)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. Micro View (Zoomed in on the origin)
x_zoom = jnp.linspace(-2, 1, 500)
axes[1].plot(x_zoom, jax.nn.relu(x_zoom), label="ReLU", color="blue", linestyle="--", linewidth=2)
axes[1].plot(x_zoom, jax.nn.gelu(x_zoom), label="GELU", color="orange", linewidth=2)
axes[1].set_title("Micro View: The 'Negative Dip' and Smooth Curve", fontsize=14)
axes[1].set_xlabel("x")
axes[1].set_ylabel("Activation f(x)")
axes[1].axhline(0, color='black', linewidth=1)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

##Part 2a: Building Nano-GPT from Scratch

The GPT architecture is a **decoder-only Transformer**. It relies on Causal Self-Attention, meaning a token at position $t$ can only attend to tokens at positions $\le t$.


###Core Mathematical Components:
1. **Queries, Keys, Values**:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

2. **Masked Scaled Dot-Product Attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\intercal}{\sqrt{d_k}} + M\right) V$$

(*Where $M$ is a lower-triangular mask of $-\infty$ to prevent looking into the future*).

In [ ]:
# --- 1. Causal Self-Attention ---
class CausalSelfAttention(eqx.Module):
    c_attn: eqx.nn.Linear
    c_proj: eqx.nn.Linear
    n_head: int
    d_model: int

    def __init__(self, d_model, n_head, key):
        self.n_head = n_head
        self.d_model = d_model
        key1, key2 = jax.random.split(key)
        # Combine Q, K, V into a single projection for efficiency
        self.c_attn = eqx.nn.Linear(d_model, 3 * d_model, key=key1)
        self.c_proj = eqx.nn.Linear(d_model, d_model, key=key2)

    def __call__(self, x):
        seq_len, _ = x.shape
        # Q, K, V projections
        qkv = jax.vmap(self.c_attn)(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        # Reshape for multi-head attention: (seq_len, n_head, head_dim) -> (n_head, seq_len, head_dim)
        head_dim = self.d_model // self.n_head
        q = q.reshape(seq_len, self.n_head, head_dim).transpose(1, 0, 2)
        k = k.reshape(seq_len, self.n_head, head_dim).transpose(1, 0, 2)
        v = v.reshape(seq_len, self.n_head, head_dim).transpose(1, 0, 2)

        # Compute Attention scores
        att = (q @ k.transpose(0, 2, 1)) * (1.0 / jnp.sqrt(head_dim))

        # Causal Masking (Lower triangular matrix)
        mask = jnp.tril(jnp.ones((seq_len, seq_len)))
        att = jnp.where(mask == 0, -1e9, att)

        att = jax.nn.softmax(att, axis=-1)

        # Aggregate values and concatenate heads
        y = (att @ v).transpose(1, 0, 2).reshape(seq_len, self.d_model)
        return jax.vmap(self.c_proj)(y)

# --- 2. Multi-Layer Perceptron (MLP) ---
class MLP(eqx.Module):
    c_fc: eqx.nn.Linear
    c_proj: eqx.nn.Linear

    def __init__(self, d_model, key):
        key1, key2 = jax.random.split(key)
        self.c_fc = eqx.nn.Linear(d_model, 4 * d_model, key=key1)
        self.c_proj = eqx.nn.Linear(4 * d_model, d_model, key=key2)

    def __call__(self, x):
        x = jax.vmap(self.c_fc)(x)
        x = jax.nn.gelu(x)
        return jax.vmap(self.c_proj)(x)

# --- 3. Transformer Block ---
class Block(eqx.Module):
    ln_1: eqx.nn.LayerNorm
    attn: CausalSelfAttention
    ln_2: eqx.nn.LayerNorm
    mlp: MLP

    def __init__(self, d_model, n_head, key):
        key1, key2 = jax.random.split(key)
        self.ln_1 = eqx.nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, key=key1)
        self.ln_2 = eqx.nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, key=key2)

    def __call__(self, x):
        # Pre-norm formulation with residual connections
        x = x + self.attn(jax.vmap(self.ln_1)(x))
        x = x + self.mlp(jax.vmap(self.ln_2)(x))
        return x

# --- 4. The Full Nano-GPT Model ---
class NanoGPT(eqx.Module):
    wte: eqx.nn.Embedding # Token embedding
    wpe: eqx.nn.Embedding # Positional embedding
    blocks: list
    ln_f: eqx.nn.LayerNorm
    lm_head: eqx.nn.Linear
    d_model: int

    def __init__(self, vocab_size, d_model, n_head, n_layer, max_seq_len, key):
        self.d_model = d_model
        keys = jax.random.split(key, 3 + n_layer)

        self.wte = eqx.nn.Embedding(vocab_size, d_model, key=keys[0])
        self.wpe = eqx.nn.Embedding(max_seq_len, d_model, key=keys[1])
        self.blocks = [Block(d_model, n_head, key=k) for k in keys[2:-1]]
        self.ln_f = eqx.nn.LayerNorm(d_model)
        self.lm_head = eqx.nn.Linear(d_model, vocab_size, use_bias=False, key=keys[-1])

    def __call__(self, idx):
        # idx shape: (seq_len,)
        seq_len = idx.shape[0]
        pos = jnp.arange(0, seq_len)

        # Add token and position embeddings
        x = jax.vmap(self.wte)(idx) + jax.vmap(self.wpe)(pos)

        # Pass through Transformer blocks
        for block in self.blocks:
            x = block(x)

        x = jax.vmap(self.ln_f)(x)
        logits = jax.vmap(self.lm_head)(x)
        return logits

In [ ]:
# We need to instantiate the model first (using the hyperparams we will use later)
# d_model = 64, n_head = 4, n_layer = 3
# Instantiate Model and Render
key = jax.random.PRNGKey(0)
# Hyperparameters: d_model=64, n_head=4, n_layer=3
gpt_model = NanoGPT(vocab_size=vocab_size, d_model=64, n_head=4, n_layer=3, max_seq_len=SEQ_LEN, key=key)


##Part 2b: Understanding and Counting Model Parameters

Before we render the architecture or train the model, let's break down exactly how many learnable parameters (weights and biases) we just created.

A "parameter" is any individual float value the model will update during backpropagation. In a GPT architecture, the parameters are distributed across embeddings, attention mechanisms, feedforward networks, and normalization layers.

###Mathematical Breakdown of Parameters

Let $d$ be the embedding dimension (`d_model`), $V$ be the vocabulary size (`vocab_size`), $L$ be the max sequence length (`max_seq_len`), and $N$ be the number of layers.

1. **Token & Position Embeddings:**

* `wte` (Token): A lookup table of size $V \times d$. $\rightarrow$ $V \times d$ parameters

* `wpe` (Position): A lookup table of size $L \times d$. $\rightarrow$ $L \times d$ parameters

2. **Inside a Single Transformer Block:**

* **LayerNorms (2x)**: Each has a learnable scale ($\gamma$) and shift ($\beta$) vector of size $d$. $\rightarrow$ $4d$ parameters

* **Causal Self-Attention**:
> * `c_attn` (QKV projection): Weight matrix $d \times 3d$, plus bias $3d$. $\rightarrow$ $3d^2 + 3d$ parameters
> * `c_proj` (Output projection): Weight matrix $d \times d$, plus bias $d$. $\rightarrow$ $d^2 + d$ parameters

* **Multi-Layer Perceptron (MLP):**

> * `c_fc` (Expansion): Weight matrix $d \times 4d$, plus bias $4d$. $\rightarrow$ $4d^2 + 4d$ parameters

>* `c_proj` (Contraction): Weight matrix $4d \times d$, plus bias $d$. $\rightarrow$ $4d^2 + d$ parameters

3. **Final Output:**

* Final LayerNorm: $\rightarrow$ $2d$ parameters

* Language Model Head: Projects from $d$ back to $V$. We omitted the bias here (standard practice since the embeddings often tie to this layer, though we left them untied for simplicity). $\rightarrow$ $d \times V$ parameters

Let's write a function to iterate through our Equinox model, isolate the arrays (the actual learnable weights), and calculate these counts dynamically.

In [ ]:
# --- Parameter Counting Utility ---
def count_parameters(model):
    """
    Accepts an Equinox Nano-GPT model and provides a detailed breakdown
    of the parameter count per sub-module, ending with the total.
    """
    # Helper function to count parameters in a specific Equinox module
    def count_module(module):
        # eqx.filter isolates only the JAX arrays (tensors) from the Python class
        arrays = jax.tree_util.tree_leaves(eqx.filter(module, eqx.is_array))
        return sum(x.size for x in arrays)

    print("="*50)
    print(f"{'Component':<30} | {'Parameter Count':>15}")
    print("="*50)

    # 1. Embeddings
    wte_params = count_module(model.wte)
    wpe_params = count_module(model.wpe)
    print(f"{'Token Embedding (wte)':<30} | {wte_params:>15,}")
    print(f"{'Position Embedding (wpe)':<30} | {wpe_params:>15,}")
    print("-" * 50)

    # 2. Transformer Blocks
    total_block_params = 0
    for i, block in enumerate(model.blocks):
        # Breakdown the block into Attention, MLP, and Norms
        attn_params = count_module(block.attn)
        mlp_params = count_module(block.mlp)
        ln1_params = count_module(block.ln_1)
        ln2_params = count_module(block.ln_2)

        block_total = attn_params + mlp_params + ln1_params + ln2_params
        total_block_params += block_total

        print(f"Block {i+1} Total                     | {block_total:>15,}")
        print(f"  -> Attention                    | {attn_params:>15,}")
        print(f"  -> MLP                          | {mlp_params:>15,}")
        print(f"  -> LayerNorms (x2)              | {ln1_params + ln2_params:>15,}")

    print("-" * 50)

    # 3. Output Head
    ln_f_params = count_module(model.ln_f)
    lm_head_params = count_module(model.lm_head)
    print(f"{'Final LayerNorm (ln_f)':<30} | {ln_f_params:>15,}")
    print(f"{'LM Head (lm_head)':<30} | {lm_head_params:>15,}")
    print("="*50)

    # 4. Total Calculation
    total_params = count_module(model)
    print(f"{'TOTAL MODEL PARAMETERS':<30} | {total_params:>15,}")
    print("="*50)

    return total_params


# Execute the counter
total_params = count_parameters(gpt_model)

##Part 3a: Dynamic Architecture Rendering

Let's visualize the architecture we just instantiated. This function introspects the Equinox PyTree model provided and maps out the dimensional transformations.

In [ ]:
def render_gpt_architecture(model: NanoGPT):
    """Dynamically renders the NanoGPT architecture based on the PyTree."""
    vocab_size = model.wte.num_embeddings
    d_model = model.d_model
    n_layer = len(model.blocks)
    n_head = model.blocks[0].attn.n_head

    fig, ax = plt.subplots(figsize=(8, 10))
    ax.axis('off')

    # Coordinates & dimensions
    center_x = 0.5
    y_curr = 0.1
    box_w = 0.6
    box_h = 0.08
    gap = 0.05

    def draw_box(y, text, color, height=box_h):
        rect = patches.Rectangle((center_x - box_w/2, y), box_w, height,
                                 linewidth=1.5, edgecolor='black', facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(center_x, y + height/2, text, ha='center', va='center', fontsize=11, weight='bold')
        return y + height

    def draw_arrow(y_start, y_end):
        ax.arrow(center_x, y_start, 0, y_end - y_start - 0.01,
                 head_width=0.03, head_length=0.015, fc='k', ec='k')

    # 1. Inputs
    ax.text(center_x, y_curr, f"Input Tokens: shape (seq_len,)", ha='center', va='center', fontsize=12)
    y_curr += 0.03
    draw_arrow(y_curr, y_curr + gap)
    y_curr += gap

    # 2. Embeddings
    emb_text = f"Token Emb + Pos Emb\n({vocab_size} -> {d_model})"
    y_curr = draw_box(y_curr, emb_text, "#ffcc99", box_h * 1.5)
    draw_arrow(y_curr, y_curr + gap)
    y_curr += gap

    # 3. Transformer Blocks
    block_start_y = y_curr
    for i in range(min(n_layer, 3)): # Draw up to 3 blocks to save space
        block_text = f"Transformer Block {i+1}\n(Attn Heads: {n_head}, d_model: {d_model})"
        y_curr = draw_box(y_curr, block_text, "#add8e6", box_h * 1.5)
        draw_arrow(y_curr, y_curr + gap)
        y_curr += gap

    if n_layer > 3:
        ax.text(center_x, y_curr - gap/2, f"... ({n_layer - 3} more blocks) ...", ha='center', va='center', style='italic')
        y_curr += gap/2

    # Group box for blocks
    blocks_h = y_curr - block_start_y - gap
    group_rect = patches.Rectangle((center_x - box_w/2 - 0.05, block_start_y - 0.02), box_w + 0.1, blocks_h + 0.04,
                                 linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=1)
    ax.add_patch(group_rect)
    ax.text(center_x + box_w/2 + 0.07, block_start_y + blocks_h/2, f"x {n_layer} Layers", va='center', weight='bold', color='gray')

    # 4. LayerNorm & Output Projection
    y_curr = draw_box(y_curr, f"Final LayerNorm", "#e0e0e0")
    draw_arrow(y_curr, y_curr + gap)
    y_curr += gap

    y_curr = draw_box(y_curr, f"LM Head (Linear)\n({d_model} -> {vocab_size})", "#90ee90", box_h * 1.5)
    draw_arrow(y_curr, y_curr + gap)
    y_curr += gap

    # 5. Output
    ax.text(center_x, y_curr, f"Logits: shape (seq_len, {vocab_size})", ha='center', va='center', fontsize=12)

    plt.title("Nano-GPT Architecture Render", fontsize=16, weight='bold')
    plt.ylim(0, y_curr + 0.05)
    plt.show()



render_gpt_architecture(gpt_model)

##Part 3b: Deep Dive - Visualizing the Attention & MLP Sublayers

To truly master the Transformer, we must understand the exact dimensional transformations occurring inside the block. The standard GPT block uses a **Pre-Norm** architecture with two main residual streams:

1. The **Attention Stream:** Where tokens look at previous tokens (mixing spatial information).

2. **The Feedforward (MLP) Stream:** Where each token is processed individually to extract deeper features (mixing feature information).

The following cell dynamically extracts the hyperparameters from our instantiated NanoGPT model and renders a high-fidelity blueprint of exactly how the tensors reshape and multiply.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def render_gpt_block_detailed(model: NanoGPT):
    """
    Renders a highly detailed diagram of a single NanoGPT Transformer Block,
    showing Q, K, V splits, Feedforward matrices, and all tensor shapes.
    """
    # Extract dimensions from the model
    d_model = model.d_model
    n_head = model.blocks[0].attn.n_head
    head_dim = d_model // n_head
    mlp_dim = 4 * d_model
    vocab_size = model.wte.num_embeddings

    fig, ax = plt.subplots(figsize=(14, 20))
    ax.axis('off')

    # Layout configuration
    center_x = 5
    box_w = 4
    box_h = 1.0

    def draw_box(x, y, w, h, title, weight_text=None, color="lightblue"):
        """Draws a component box with optional weight matrix dimensions."""
        rect = patches.FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.1",
                                      linewidth=2, edgecolor='black', facecolor=color, zorder=3)
        ax.add_patch(rect)

        ax.text(x, y + (0.15 if weight_text else 0), title, ha='center', va='center', fontsize=12, weight='bold')
        if weight_text:
            ax.text(x, y - 0.25, weight_text, ha='center', va='center', fontsize=10, family='monospace', color='#333333')

    def draw_tensor_arrow(x1, y1, x2, y2, tensor_shape, align='center'):
        """Draws an arrow and labels the shape of the tensor flowing through it."""
        ax.arrow(x1, y1, x2-x1, y2-y1, head_width=0.15, head_length=0.2, fc='k', ec='k', length_includes_head=True, zorder=2)

        # Calculate text position
        tx = x1 + (x2 - x1) / 2
        ty = y1 + (y2 - y1) / 2

        # Offset text slightly so it doesn't overlap the line
        offset = 0.3 if align == 'left' else (-0.3 if align == 'right' else 0.2)
        ax.text(tx + (offset if x1==x2 else 0), ty, f"Shape: {tensor_shape}",
                ha=('left' if align=='left' else 'right' if align=='right' else 'left'),
                va='center', fontsize=10, color='blue', rotation=0 if x1==x2 else 90)

    def draw_residual(y_start, y_end, target_y):
        """Draws a skip connection path."""
        res_x = 1.5
        ax.plot([center_x, res_x], [y_start, y_start], 'k-', lw=2, zorder=1) # Left
        ax.plot([res_x, res_x], [y_start, target_y], 'k-', lw=2, zorder=1)   # Down
        ax.arrow(res_x, target_y, center_x - res_x - 0.3, 0, head_width=0.15, head_length=0.2, fc='k', ec='k', lw=2, zorder=1) # Right to Add node

    # ==========================================
    # 1. Block Boundary
    # ==========================================
    # Draw a massive dashed bounding box for the Transformer Block
    block_rect = patches.Rectangle((center_x - 4.5, 6), 9, 21, linewidth=3, edgecolor='gray', linestyle='--', facecolor='#f9f9f9', zorder=0)
    ax.add_patch(block_rect)
    ax.text(center_x - 4.2, 26.5, "Single Transformer Block", fontsize=16, weight='bold', color='gray')

    # ==========================================
    # 2. Input to the Block
    # ==========================================
    y = 28
    ax.text(center_x, y+0.5, "Input from Embeddings (or Prev Layer)", ha='center', va='center', fontsize=12, weight='bold')
    draw_tensor_arrow(center_x, y, center_x, 26, f"[seq_len, {d_model}]", align='right')

    # ==========================================
    # 3. Attention Sublayer
    # ==========================================
    y = 25
    draw_box(center_x, y, box_w, box_h, "LayerNorm 1", color="#e0e0e0")
    draw_residual(26, 26, 15) # Residual stream 1 starts here, targets Add 1 at y=15
    draw_tensor_arrow(center_x, y - box_h/2, center_x, y - 1.5, f"[seq_len, {d_model}]", align='right')

    y = 22.5
    draw_box(center_x, y, box_w + 1, box_h * 1.2, "QKV Linear Projection (c_attn)",
             f"Weight W_qkv: [{d_model}, 3 * {d_model}]", color="#ffcc99")
    draw_tensor_arrow(center_x, y - box_h*0.6, center_x, y - 1.5, f"[seq_len, {3 * d_model}]", align='right')

    # Split into Q, K, V
    y = 20
    q_x, k_x, v_x = center_x - 2.5, center_x, center_x + 2.5

    # Split arrows
    ax.arrow(center_x, 21, q_x - center_x, y - 21 + box_h/2, head_width=0.1, color='k', length_includes_head=True)
    ax.arrow(center_x, 21, 0, y - 21 + box_h/2, head_width=0.1, color='k', length_includes_head=True)
    ax.arrow(center_x, 21, v_x - center_x, y - 21 + box_h/2, head_width=0.1, color='k', length_includes_head=True)
    ax.text(center_x + 0.5, 20.8, "Split & Reshape", fontsize=9, style='italic', color='red')

    draw_box(q_x, y, 2, box_h, "Query (Q)", f"[{n_head}, seq_len, {head_dim}]", color="#ffb3b3")
    draw_box(k_x, y, 2, box_h, "Key (K)", f"[{n_head}, seq_len, {head_dim}]", color="#b3d9ff")
    draw_box(v_x, y, 2, box_h, "Value (V)", f"[{n_head}, seq_len, {head_dim}]", color="#d9b3ff")

    # Masked Attention Output
    y = 17.5
    draw_box(center_x, y, box_w + 2, box_h, "Masked Scaled Dot-Product Attention\nsoftmax( (Q @ K.T)/sqrt(d) + Mask ) @ V", color="#add8e6")

    # Connect Q, K, V to Attention
    ax.plot([q_x, q_x], [19.5, 18], 'k-', zorder=1)
    ax.plot([v_x, v_x], [19.5, 18], 'k-', zorder=1)
    ax.arrow(q_x, 18, center_x - q_x, -0.5, head_width=0.1, fc='k', ec='k', length_includes_head=True)
    ax.arrow(v_x, 18, center_x - v_x, -0.5, head_width=0.1, fc='k', ec='k', length_includes_head=True)
    ax.arrow(k_x, 19.5, 0, -1.5, head_width=0.1, fc='k', ec='k', length_includes_head=True)

    draw_tensor_arrow(center_x, y - box_h/2, center_x, y - 1.5, f"[seq_len, {d_model}]", align='right')

    y = 15
    # Add Node 1
    ax.add_patch(patches.Circle((center_x, y), 0.3, fc='#ffffb3', ec='black', zorder=4))
    ax.text(center_x, y, "+", ha='center', va='center', fontsize=16, weight='bold', zorder=5)
    ax.text(center_x + 0.5, y, "Residual Add", ha='left', va='center', fontsize=11, weight='bold')

    # ==========================================
    # 4. Feedforward (MLP) Sublayer
    # ==========================================
    draw_tensor_arrow(center_x, y - 0.3, center_x, y - 1.5, f"[seq_len, {d_model}]", align='right')

    y = 13
    draw_box(center_x, y, box_w, box_h, "LayerNorm 2", color="#e0e0e0")
    draw_residual(14, 14, 7) # Residual stream 2 starts here, targets Add 2 at y=7
    draw_tensor_arrow(center_x, y - box_h/2, center_x, y - 1.5, f"[seq_len, {d_model}]", align='right')

    y = 11
    draw_box(center_x, y, box_w + 1, box_h * 1.2, "MLP Hidden (c_fc) + GELU",
             f"Weight W_1: [{d_model}, {mlp_dim}]", color="#98fb98")

    # Wide matrix representation for MLP dimension expansion
    ax.plot([center_x - box_w/2 - 0.5, center_x - box_w/2 - 0.5], [y-0.4, y+0.4], 'k-', lw=3)
    ax.plot([center_x + box_w/2 + 0.5, center_x + box_w/2 + 0.5], [y-0.4, y+0.4], 'k-', lw=3)

    draw_tensor_arrow(center_x, y - box_h*0.6, center_x, y - 1.5, f"[seq_len, {mlp_dim}]", align='right')

    y = 9
    draw_box(center_x, y, box_w + 1, box_h * 1.2, "MLP Projection (c_proj)",
             f"Weight W_2: [{mlp_dim}, {d_model}]", color="#98fb98")
    draw_tensor_arrow(center_x, y - box_h*0.6, center_x, y - 1.5, f"[seq_len, {d_model}]", align='right')

    y = 7
    # Add Node 2
    ax.add_patch(patches.Circle((center_x, y), 0.3, fc='#ffffb3', ec='black', zorder=4))
    ax.text(center_x, y, "+", ha='center', va='center', fontsize=16, weight='bold', zorder=5)
    ax.text(center_x + 0.5, y, "Residual Add", ha='left', va='center', fontsize=11, weight='bold')

    # ==========================================
    # 5. Output to Next Layer / LM Head
    # ==========================================
    draw_tensor_arrow(center_x, y - 0.3, center_x, y - 2.5, f"Interface to Next Layer:\n[seq_len, {d_model}]", align='right')

    y = 3
    draw_box(center_x, y, box_w + 1, box_h * 1.2, "Final LM Head (If last block)",
             f"Weight W_head: [{d_model}, {vocab_size}]", color="#ffb6c1")
    draw_tensor_arrow(center_x, y - box_h*0.6, center_x, y - 2, f"[seq_len, {vocab_size}] (Logits)", align='right')

    plt.title(f"Detailed Transformer Block Blueprint\n(d_model={d_model}, n_head={n_head}, vocab={vocab_size})", fontsize=18, weight='bold', y=1.02)
    plt.xlim(0, 10)
    plt.ylim(0, 29)
    plt.show()

# Render the detailed blueprint based on our trained NanoGPT
render_gpt_block_detailed(gpt_model)

## Part 4: Training & Evaluation with Optax

We will define a batched cross-entropy loss function. Since our model operates on single sequences, we use jax.vmap over the model to efficiently process batches of data.

In [ ]:
# --- Training Configuration ---
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 3e-4

optimizer = optax.adamw(LEARNING_RATE)
opt_state = optimizer.init(eqx.filter(gpt_model, eqx.is_array))

# --- Loss Function ---
@eqx.filter_value_and_grad
def compute_loss(model, x_batch, y_batch):
    # Vectorize forward pass over the batch
    batched_forward = jax.vmap(model)
    logits = batched_forward(x_batch) # Shape: (batch, seq_len, vocab_size)

    # Flatten batches and sequences for cross entropy calculation
    logits_flat = logits.reshape(-1, vocab_size)
    y_flat = y_batch.reshape(-1)

    loss = optax.softmax_cross_entropy_with_integer_labels(logits_flat, y_flat)
    return jnp.mean(loss)

# --- Training Step ---
@eqx.filter_jit
def train_step(model, opt_state, x_batch, y_batch):
    loss, grads = compute_loss(model, x_batch, y_batch)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def eval_step(model, x_batch, y_batch):
    batched_forward = jax.vmap(model)
    logits = batched_forward(x_batch)
    logits_flat = logits.reshape(-1, vocab_size)
    y_flat = y_batch.reshape(-1)
    return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits_flat, y_flat))

# --- Training Loop ---
print("Training Nano-GPT...")
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # Shuffle training data
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    Y_train_shuffled = Y_train[indices]

    epoch_loss = 0.0
    n_batches = max(1, len(X_train) // BATCH_SIZE)

    for i in range(n_batches):
        bx = X_train_shuffled[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        by = Y_train_shuffled[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        gpt_model, opt_state, loss = train_step(gpt_model, opt_state, bx, by)
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / n_batches
    val_loss = eval_step(gpt_model, X_val, Y_val).item()

    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")

# --- Plot Losses ---
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss', color='blue', lw=2)
plt.plot(val_losses, label='Validation Loss', color='orange', lw=2)
plt.title("Nano-GPT Cross-Entropy Loss over Epochs", fontsize=14)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

##Part 5: Interactive Text Generation

Now for the fun part! Language models generate text **autoregressively**: they predict the next token, append it to the context, and repeat the process.

Below is an interactive generator. Feel free to alter the prompt string to see how the network attempts to complete the story based on what it learned.

In [ ]:
# --- Autoregressive Generation ---
@eqx.filter_jit
def generate_next_token(model, context_idx):
    # Get logits for the sequence
    logits = model(context_idx)
    # We only care about the prediction for the very last step
    next_token_logits = logits[-1, :]
    # Greedy decoding: pick the highest probability token
    next_token = jnp.argmax(next_token_logits)
    return next_token

def generate_text(model, prompt, max_new_tokens=50):
    # Encode prompt
    context = encode(prompt)

    print(f"--- Generating Text ---")
    print(prompt, end="")

    for _ in range(max_new_tokens):
        # Crop context to max sequence length to avoid out-of-bounds pos embeddings
        idx_cond = jnp.array(context[-SEQ_LEN:])

        # Predict next token
        next_token = generate_next_token(model, idx_cond).item()

        # Append to context
        context.append(next_token)

        # Print stream
        print(itos[next_token], end="", flush=True)

    print("\n-----------------------")

# ---------------------------------------------------------
# INTERACTIVE PROMPT
# Change this string to test the model's generation!
# ---------------------------------------------------------
user_prompt = "The hare"

generate_text(gpt_model, user_prompt, max_new_tokens=500)

##Part 6: Peeking Inside the Black Box - Visualizing Attention


In a Transformer, the "magic" happens in the self-attention mechanism. When the model processes a token at position $t$, it queries all previous tokens (positions $\le t$) to gather context.

Because we are building a Causal Language Model (GPT), a token is strictly forbidden from looking at future tokens. We enforce this by applying a lower-triangular mask of $-\infty$ to the attention scores before the softmax step.

In this section, we will extract the intermediate attention probabilities from our trained NanoGPT and plot them as heatmaps for a 32-token sequence.

* Y-axis (Rows): The current token generating a Query.
* X-axis (Columns): The past tokens providing Keys/Values.
* Gray Area: The masked future tokens that the model is prevented from attending to.

In [ ]:
# --- 1. Extract Attention Maps from the Trained Model ---
def extract_attention_maps(model, text_prompt, max_len=32):
    """
    Runs a forward pass and manually extracts the attention matrices
    from each Transformer block.
    """
    # Encode up to max_len tokens
    idx = jnp.array(encode(text_prompt)[:max_len])
    seq_len = idx.shape[0]

    # Extract the character strings for our axis labels (replace space with _)
    chars = [itos[i.item()] if itos[i.item()] != ' ' else '_' for i in idx]

    # 1. Embeddings
    pos = jnp.arange(0, seq_len)
    x = jax.vmap(model.wte)(idx) + jax.vmap(model.wpe)(pos)

    attention_maps = []

    # 2. Iterate through blocks to intercept the attention scores
    for block in model.blocks:
        # Pre-layer norm
        ln_x = jax.vmap(block.ln_1)(x)

        # Extract Q, K, V
        qkv = jax.vmap(block.attn.c_attn)(ln_x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        n_head = block.attn.n_head
        head_dim = block.attn.d_model // n_head

        # Reshape for multi-head: (seq_len, n_head, head_dim) -> (n_head, seq_len, head_dim)
        q = q.reshape(seq_len, n_head, head_dim).transpose(1, 0, 2)
        k = k.reshape(seq_len, n_head, head_dim).transpose(1, 0, 2)

        # Calculate raw attention scores
        att = (q @ k.transpose(0, 2, 1)) * (1.0 / jnp.sqrt(head_dim))

        # Apply the causal mask
        mask = jnp.tril(jnp.ones((seq_len, seq_len)))
        att = jnp.where(mask == 0, -1e9, att)

        # Softmax to get probabilities (0.0 to 1.0)
        att_probs = jax.nn.softmax(att, axis=-1)

        # Average the attention probabilities across all heads for visualization
        avg_att_probs = jnp.mean(att_probs, axis=0)
        attention_maps.append(np.array(avg_att_probs))

        # Continue the standard forward pass to get input for the next block
        x = block(x)

    return attention_maps, chars

# --- 2. Render the Heatmaps ---
def plot_attention_heatmaps(attention_maps, chars):
    n_layers = len(attention_maps)
    seq_len = len(chars)

    fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 6))
    if n_layers == 1: axes = [axes]

    # Create a colormap (Plasma) and explicitly set NaN values to Gray
    cmap = plt.cm.plasma.copy()
    cmap.set_bad(color='lightgray')

    # Create an upper-triangular mask (future tokens) to set them to NaN
    future_mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)

    for i, (ax, att_map) in enumerate(zip(axes, attention_maps)):
        plot_data = np.copy(att_map)
        plot_data[future_mask] = np.nan # Apply gray mask to future

        # Plot heatmap
        im = ax.imshow(plot_data, cmap=cmap, vmin=0, vmax=np.max(att_map))

        ax.set_title(f"Layer {i+1} Attention", fontsize=14, weight='bold')
        ax.set_xticks(np.arange(seq_len))
        ax.set_yticks(np.arange(seq_len))

        # Set character labels
        ax.set_xticklabels(chars, rotation=90, fontsize=9, family='monospace')
        ax.set_yticklabels(chars, fontsize=9, family='monospace')

        ax.set_xlabel("Attending To (Key)", fontsize=11)
        if i == 0:
            ax.set_ylabel("Current Token (Query)", fontsize=11)

        # Add colorbar
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Attention Probability")

    plt.suptitle(f"Causal Self-Attention Heatmaps (Averaged across {gpt_model.blocks[0].attn.n_head} Heads)", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Let's extract exactly 32 tokens from our training text
# to see how the model processes a full sequence block.
# ---------------------------------------------------------
sample_text = text[1:33] # "The Hare and the Tortoise.\nA Har"
print(f"Visualizing Attention for string: '{sample_text}'\n")

att_maps, tokens = extract_attention_maps(gpt_model, sample_text, max_len=32)
plot_attention_heatmaps(att_maps, tokens)

##Part 7: Understanding "Hallucinations" and Mechanistic Interpretability

As you interact with the text generation cell, you might notice the model occasionally outputs complete nonsense or confidently states things that did not happen in the training text. In the context of Large Language Models (LLMs), this phenomenon is commonly referred to as **hallucination**.


###Why do models hallucinate?
It is crucial to remember that autoregressive Transformers are not databases querying stored facts. They are probabilistic engines.

1. **Next-Token Prediction:** The model is simply sampling from a probability distribution ($P(x_t | \text{context})$). If the context leads to a state where multiple tokens have similar probabilities, the model might sample a path that is grammatically correct but factually wrong.

2. **Interpolation in Latent Space:** If the model encounters a prompt that is very different from its training data (out-of-distribution), it will interpolate based on the closest statistical patterns it knows, often stitching together unrelated concepts.

3. **Attention Dilution:** In long contexts, the attention mechanism might "forget" or dilute the weight of crucial early tokens, causing the model to lose the thread of the narrative and generate inconsistent continuations.

###How investigating activations helps

To figure out why a model hallucinated, researchers use a field of study called **Mechanistic Interpretability**. Instead of treating the network as a black box, we look at the internal activations (the outputs of specific neurons) for a single prediction.

* **Attention Maps:** As we saw in Part 6, if the model hallucinates the wrong subject for a verb, looking at the attention map might reveal that the token was attending to the wrong noun in the prompt.

* **MLP Activations (The "Memory" Bank)**: The Feedforward (MLP) layers in a Transformer act like key-value memory stores. The first linear projection expands the dimension (often by 4x), creating thousands of individual neurons. By looking at which specific neurons "fire" (have high activation values) after the GELU non-linearity, we can often map specific neurons to specific concepts, characters, or grammatical rules.

If the model hallucinates that the "Tortoise" is fast, investigating the MLP activations might show that the "speed" neuron fired when it shouldn't have.Let's write a tool to extract and visualize these MLP activations for the very last token in our prompt—the token responsible for predicting the next generated character!

In [ ]:
# --- 1. Extract MLP Activations ---
def extract_mlp_activations(model, text_prompt):
    """
    Runs a forward pass and intercepts the internal MLP hidden state
    (after the GELU activation) for the final token in the prompt.
    """
    idx = jnp.array(encode(text_prompt))
    seq_len = idx.shape[0]
    pos = jnp.arange(0, seq_len)

    # 1. Embeddings
    x = jax.vmap(model.wte)(idx) + jax.vmap(model.wpe)(pos)

    layer_activations = []

    # 2. Iterate through blocks to intercept MLP states
    for block in model.blocks:
        # --- Attention Stream ---
        attn_out = block.attn(jax.vmap(block.ln_1)(x))
        x = x + attn_out

        # --- MLP Stream ---
        mlp_input = jax.vmap(block.ln_2)(x)

        # Step A: First linear projection (c_fc) -> expands dimension to 4 * d_model
        hidden = jax.vmap(block.mlp.c_fc)(mlp_input)

        # Step B: Non-linearity (GELU) -> This is the activation we want to see!
        hidden_act = jax.nn.gelu(hidden)

        # We only care about the activations for the LAST token in the sequence,
        # because this is the token driving the generation of the next character.
        last_token_acts = hidden_act[-1, :]
        layer_activations.append(np.array(last_token_acts))

        # Step C: Complete the block forward pass
        mlp_out = jax.vmap(block.mlp.c_proj)(hidden_act)
        x = x + mlp_out

    # Return shape: (n_layers, 4 * d_model)
    return np.array(layer_activations)

# --- 2. Visualize the Activations ---
def plot_mlp_activations(activations, prompt_text):
    n_layers, mlp_dim = activations.shape

    fig, ax = plt.subplots(figsize=(12, 4))

    # Plot as a heatmap. We use the 'hot' colormap where black=0 and white/yellow=high activation
    im = ax.imshow(activations, cmap='hot', aspect='auto', interpolation='nearest')

    ax.set_title(f"MLP Neuron Activations for the token predicting what comes after: '{prompt_text}'", fontsize=13)
    ax.set_xlabel(f"Neuron Index (0 to {mlp_dim-1})", fontsize=11)
    ax.set_ylabel("Transformer Layer", fontsize=11)

    # Clean up y-ticks to show layer numbers
    ax.set_yticks(np.arange(n_layers))
    ax.set_yticklabels([f"Layer {i+1}" for i in range(n_layers)])

    fig.colorbar(im, ax=ax, label="Activation Magnitude (GELU output)")
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Test it out!
# Let's see which neurons fire when the model tries to complete "The Hare was "
# ---------------------------------------------------------
test_prompt = "The Hare was "

print(f"Extracting memory activations for the prompt: '{test_prompt}'")

mlp_acts = extract_mlp_activations(gpt_model, test_prompt)
plot_mlp_activations(mlp_acts, test_prompt)